# Opcode-Based OpenVPN Detection

This notebook detects OpenVPN by parsing **control channel opcodes** from UDP traffic on port 1194.

## How it works
1. Read all packets from a PCAP file
2. Filter for UDP packets on port 1194 with payload data
3. Extract the opcode from the first byte (top 5 bits)
4. Match the sequence against the expected OpenVPN handshake pattern

## OpenVPN Control Channel Opcodes
| Value | Constant | Description |
|-------|----------|-------------|
| 1 | P_CONTROL_HARD_RESET_CLIENT_V2 | Client initiates connection |
| 2 | P_CONTROL_HARD_RESET_SERVER_V2 | Server responds with reset |
| 3 | P_CONTROL_SOFT_RESET_V1 | Soft key refresh |
| 4 | P_CONTROL_V1 | Control message (TLS handshake data) |
| 5 | P_ACK_V1 | Acknowledgement |
| 6 | P_DATA_V1 | Data channel packet (encrypted) |
| 7 | P_CONTROL_HARD_RESET_CLIENT_V1 | Legacy client reset |
| 8 | P_CONTROL_HARD_RESET_SERVER_V1 | Legacy server reset |
| 9 | P_DATA_V2 | Data channel packet (v2) |

## Expected Handshake Pattern
```
HARD_RESET -> CONTROL -> ACK -> DATA
```
This sequence indicates a successful OpenVPN negotiation followed by encrypted data transfer.

In [ ]:
from scapy.all import rdpcap, Raw, UDP
import os

OPENVPN_PORT = 1194

def extract_opcode(byte):
    """Extract the 5-bit opcode from the first byte of an OpenVPN packet."""
    return (byte & 0b11111000) >> 3

# OpenVPN opcode groups
HARD_RESET = {1, 2, 7, 8, 10}  # All reset types (client/server, v1/v2/v3)
CONTROL = {3, 4}               # Soft reset + control messages
ACK = {5}                      # Acknowledgement
DATA = {6, 9}                  # Data channel packets (v1 and v2)

def is_likely_openvpn(pkt):
    """Check if a packet is a potential OpenVPN packet (UDP port 1194)."""
    return (
        UDP in pkt and Raw in pkt and
        (pkt[UDP].sport == OPENVPN_PORT or pkt[UDP].dport == OPENVPN_PORT) and
        len(pkt[Raw].load) >= 1
    )

def analyze_openvpn_pattern(pcap_file):
    """Analyze PCAP for OpenVPN handshake pattern."""
    packets = rdpcap(pcap_file)
    opcodes = []

    for pkt in packets:
        if is_likely_openvpn(pkt):
            byte = pkt[Raw].load[0]
            opcode = extract_opcode(byte)
            opcodes.append(opcode)

    print(f"\n[*] Total OpenVPN-candidate packets: {len(opcodes)}")
    print(f"[*] Unique opcodes found: {sorted(set(opcodes))}")

    # State machine to detect handshake pattern
    phase = "RESET"
    reset_seen = control_seen = ack_seen = data_seen = False

    for op in opcodes:
        if op in HARD_RESET and not reset_seen:
            reset_seen = True
            continue
        elif op in CONTROL and reset_seen and not control_seen:
            control_seen = True
            continue
        elif op in ACK and control_seen and not ack_seen:
            ack_seen = True
            continue
        elif op in DATA and ack_seen:
            data_seen = True
            break

    print(f"\n[*] Handshake phases detected:")
    print(f"    RESET:  {reset_seen}")
    print(f"    CONTROL: {control_seen}")
    print(f"    ACK:     {ack_seen}")
    print(f"    DATA:    {data_seen}")

    if reset_seen and control_seen and ack_seen and data_seen:
        print("\n[+] Pattern Match: OpenVPN-like flow DETECTED.")
        print("    Pattern: HARD_RESET -> CONTROL -> ACK -> DATA")
        return True
    else:
        print("\n[-] Pattern does NOT match standard OpenVPN handshake.")
        return False

## Analyze a PCAP File

The dataset path is relative to the project root. Modify the path below to test different captures.

Try these paths:
- `data/VPN data sets/43.204.236.255/vpn traffic.pcap` (should detect OpenVPN)
- `data/VPN data sets/43.204.236.255/https-traffic.pcap` (should NOT detect OpenVPN)

In [ ]:
# Use relative path from project root
pcap_file = os.path.join("data", "VPN data sets", "43.204.236.255", "vpn traffic.pcap")

# Verify file exists
if not os.path.exists(pcap_file):
    print(f"ERROR: File not found: {pcap_file}")
    print("Make sure you have extracted 'VPN data sets.zip' into the 'data/' folder.")
else:
    analyze_openvpn_pattern(pcap_file)